# Universal Player Classification (Outfield & Goalkeepers).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
df_outfield = pd.read_csv('../data/clean/clean_df_outfield.csv')
df_gk = pd.read_csv('../data/clean/clean_df_gk.csv')
print(f"Total outfield players loaded: {len(df_outfield)}")
print(f"Total goalkeepers loaded: {len(df_gk)}")

Total outfield players loaded: 772
Total goalkeepers loaded: 52


In [2]:
import importlib
import scripts.classify
import scripts.config

In [3]:
from scripts.classify import *
from scripts.config import *

In [4]:
final_outfield_df = classify_players(df_outfield, outfield_config, hybrid_threshold=5.0)
final_gk_df       = classify_players(df_gk,       gk_config,       hybrid_threshold=5.0)

final_outfield_df.to_csv('outfield_classified.csv', index=False)
final_gk_df.to_csv('gk_classified.csv', index=False)

In [5]:
print("Outfield Tactical Role Distribution:\n")
print(final_outfield_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

print("Goalkeeper Tactical Role Distribution:\n")
print(final_gk_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

Outfield Tactical Role Distribution:

Tactical_Role       Aerial Stopper  Aerial Stopper / Ball Playing Defender  \
Broad_Position                                                               
Attacking Midfield               0                                       0   
Central Midfield                 0                                       0   
Centre-Back                     41                                       2   
Defensive Midfield               0                                       0   
Full-Back                        0                                       0   
Striker                          0                                       0   
Winger                           0                                       0   

Tactical_Role       Aerial Stopper / Defensive Sweeper  Anchor Man  \
Broad_Position                                                       
Attacking Midfield                                   0           0   
Central Midfield                                 

In [6]:
final_outfield_df.to_csv('../data/clean/classified_outfield_players.csv', index=False)
final_gk_df.to_csv('../data/clean/classified_goalkeepers.csv', index=False)
print("Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.")

Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.


# CLASSIFY PLAYERS INTO 7 POSITION CATEGORIES

In [46]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [47]:
# Load your classified data
df = pd.read_csv('../data/clean/classified_outfield_players.csv')

# Display current broad positions
print("Current Broad_Position categories:")
print(df['Broad_Position'].unique())
print(f"\nTotal players: {len(df)}")

Current Broad_Position categories:
<StringArray>
[       'Centre-Back',          'Full-Back', 'Defensive Midfield',
   'Central Midfield', 'Attacking Midfield',             'Winger',
            'Striker']
Length: 7, dtype: str

Total players: 772


In [50]:
def classify_position_7(broad_position, position):
    """
    Classify players into 7 standardized position categories:
    - Centre-Back
    - Full-Back (includes Left-Back and Right-Back - keeps original distinction)
    - Defensive Midfield
    - Central Midfield
    - Attacking Midfield
    - Winger (includes Left Winger and Right Winger)
    - Striker (includes Centre-Forward and Second Striker)
    """
    
    # Centre-Back
    if broad_position == 'Centre-Back' or position == 'Centre-Back':
        return 'Centre-Back'
    
    # Full-Back (includes Left-Back and Right-Back)
    # We keep the original position name in a separate column
    if broad_position == 'Full-Back' or position in ['Left-Back', 'Right-Back']:
        return 'Full-Back'
    
    # Defensive Midfield
    if broad_position == 'Defensive Midfield':
        return 'Defensive Midfield'
    
    # Central Midfield
    if broad_position == 'Central Midfield':
        return 'Central Midfield'
    
    # Attacking Midfield
    if broad_position == 'Attacking Midfield':
        return 'Attacking Midfield'
    
    # Winger (includes Left Winger and Right Winger)
    if broad_position == 'Winger' or position in ['Left Winger', 'Right Winger']:
        return 'Winger'
    
    # Striker (includes Centre-Forward and Second Striker)
    if broad_position == 'Striker' or position in ['Centre-Forward', 'Second Striker']:
        return 'Striker'
    
    # Default - should not happen
    return 'Other'

# Apply classification
df['Position_7'] = df.apply(
    lambda row: classify_position_7(row['Broad_Position'], row['Position']), 
    axis=1
)

# ============================================
# CREATE FULL-BACK SUB-CATEGORY COLUMN
# ============================================

def get_fullback_type(row):
    """
    Determine if a full-back is Left-Back or Right-Back
    """
    if row['Position_7'] == 'Full-Back':
        if 'Left-Back' in str(row['Position']):
            return 'Left-Back'
        elif 'Right-Back' in str(row['Position']):
            return 'Right-Back'
        elif 'Left' in str(row['Broad_Position']):
            return 'Left-Back'
        elif 'Right' in str(row['Broad_Position']):
            return 'Right-Back'
        else:
            return 'Full-Back (Unspecified)'
    return None

# Add full-back specific column
df['FullBack_Type'] = df.apply(get_fullback_type, axis=1)

# Also add winger sub-category for consistency
def get_winger_type(row):
    """
    Determine if a winger is Left Winger or Right Winger
    """
    if row['Position_7'] == 'Winger':
        if 'Left Winger' in str(row['Position']):
            return 'Left Winger'
        elif 'Right Winger' in str(row['Position']):
            return 'Right Winger'
        elif 'Left' in str(row['Broad_Position']):
            return 'Left Winger'
        elif 'Right' in str(row['Broad_Position']):
            return 'Right Winger'
        else:
            return 'Winger (Unspecified)'
    return None

df['Winger_Type'] = df.apply(get_winger_type, axis=1)

In [39]:
# ============================================
# VERIFY CLASSIFICATION
# ============================================

In [52]:
print("\n" + "="*70)
print("7 POSITION CATEGORIES DISTRIBUTION")
print("="*70)
position_counts = df['Position_7'].value_counts()
print(position_counts)
print(f"\nTotal: {position_counts.sum()} players")

# Show Full-Back breakdown
print("\n" + "="*70)
print("FULL-BACK BREAKDOWN")
print("="*70)
fullback_breakdown = df[df['Position_7'] == 'Full-Back']['FullBack_Type'].value_counts()
print(fullback_breakdown)

# Show Winger breakdown
print("\n" + "="*70)
print("WINGER BREAKDOWN")
print("="*70)
winger_breakdown = df[df['Position_7'] == 'Winger']['Winger_Type'].value_counts()
print(winger_breakdown)

# Check if any "Other" category exists
if 'Other' in position_counts.index:
    print("\nWARNING: 'Other' category found!")
    print(df[df['Position_7'] == 'Other'][['Name', 'Broad_Position', 'Position']].head())


7 POSITION CATEGORIES DISTRIBUTION
Position_7
Centre-Back           149
Full-Back             145
Winger                130
Central Midfield      115
Striker               111
Defensive Midfield     68
Attacking Midfield     54
Name: count, dtype: int64

Total: 772 players

FULL-BACK BREAKDOWN
FullBack_Type
Right-Back    77
Left-Back     68
Name: count, dtype: int64

WINGER BREAKDOWN
Winger_Type
Right Winger            66
Left Winger             62
Winger (Unspecified)     2
Name: count, dtype: int64


In [53]:
# ============================================
# FUNCTION TO REMOVE NULL COLUMNS AFTER SPLITTING
# ============================================

In [60]:
def clean_position_table(df, position_name):
    """
    Remove columns that are completely null (100% missing) for a specific position
    Also removes columns that are constant (all same value) and rows that are completely empty
    Returns: (cleaned_dataframe, list_of_removed_columns)
    """
    original_rows = len(df)
    original_cols = len(df.columns)
    
    # Make a copy to avoid modifying original
    df_clean = df.copy()
    
    # Keep track of all removed columns
    all_removed = []
    
    # Step 1: Remove columns that are 100% null
    cols_all_null = df_clean.columns[df_clean.isna().all()].tolist()
    all_removed.extend(cols_all_null)
    df_clean = df_clean.drop(columns=cols_all_null)
    
    # Step 2: Remove constant columns (all same value, excluding nulls)
    constant_cols = []
    for col in df_clean.columns:
        # Skip columns that should keep their distinction
        if col in ['FullBack_Type', 'Winger_Type', 'Position']:
            continue
        # Get unique non-null values
        unique_vals = df_clean[col].dropna().unique()
        # If only one unique value (or all nulls), it's constant
        if len(unique_vals) <= 1:
            constant_cols.append(col)
    all_removed.extend(constant_cols)
    df_clean = df_clean.drop(columns=constant_cols)
    
    # Step 3: Remove rows that are completely empty
    before_rows = len(df_clean)
    df_clean = df_clean.dropna(how='all')
    rows_removed = before_rows - len(df_clean)
    
    # Step 4: Remove columns that are now 100% null after row removal (safety check)
    cols_all_null_after = df_clean.columns[df_clean.isna().all()].tolist()
    all_removed.extend(cols_all_null_after)
    df_clean = df_clean.drop(columns=cols_all_null_after)
    
    # Print cleaning report
    print(f"\n{position_name}:")
    print(f"   Original: {original_rows} rows, {original_cols} columns")
    print(f"   Removed 100% null columns: {len(cols_all_null)}")
    print(f"   Removed constant columns: {len(constant_cols)}")
    print(f"   Removed {rows_removed} empty rows")
    print(f"   After cleaning: {len(df_clean)} rows, {len(df_clean.columns)} columns")
    print(f"   Total columns removed: {original_cols - len(df_clean.columns)}")
    
    # Show sample of removed columns
    if all_removed and len(all_removed) <= 20:
        print(f"   Removed columns: {all_removed}")
    elif all_removed:
        print(f"   Sample removed columns: {all_removed[:10]}{'...' if len(all_removed) > 10 else ''}")
    
    # Return both the cleaned dataframe AND the list of removed columns
    return df_clean, all_removed

In [61]:
# ============================================
# SPLIT DATA BY POSITION (BEFORE CLEANING)
# ============================================

In [62]:
# Create raw position dataframes (before cleaning)
centre_backs_raw = df[df['Position_7'] == 'Centre-Back'].copy()
full_backs_raw = df[df['Position_7'] == 'Full-Back'].copy()
defensive_midfielders_raw = df[df['Position_7'] == 'Defensive Midfield'].copy()
central_midfielders_raw = df[df['Position_7'] == 'Central Midfield'].copy()
attacking_midfielders_raw = df[df['Position_7'] == 'Attacking Midfield'].copy()
wingers_raw = df[df['Position_7'] == 'Winger'].copy()
strikers_raw = df[df['Position_7'] == 'Striker'].copy()

print(f"Centre-Backs:          {len(centre_backs_raw)} players, {len(centre_backs_raw.columns)} columns")
print(f"Full-Backs:            {len(full_backs_raw)} players, {len(full_backs_raw.columns)} columns")
print(f"  - Left-Backs:        {len(full_backs_raw[full_backs_raw['FullBack_Type'] == 'Left-Back'])}")
print(f"  - Right-Backs:       {len(full_backs_raw[full_backs_raw['FullBack_Type'] == 'Right-Back'])}")
print(f"Defensive Midfielders: {len(defensive_midfielders_raw)} players, {len(defensive_midfielders_raw.columns)} columns")
print(f"Central Midfielders:   {len(central_midfielders_raw)} players, {len(central_midfielders_raw.columns)} columns")
print(f"Attacking Midfielders: {len(attacking_midfielders_raw)} players, {len(attacking_midfielders_raw.columns)} columns")
print(f"Wingers:               {len(wingers_raw)} players, {len(wingers_raw.columns)} columns")
print(f"  - Left Wingers:      {len(wingers_raw[wingers_raw['Winger_Type'] == 'Left Winger'])}")
print(f"  - Right Wingers:     {len(wingers_raw[wingers_raw['Winger_Type'] == 'Right Winger'])}")
print(f"Strikers:              {len(strikers_raw)} players, {len(strikers_raw.columns)} columns")

Centre-Backs:          149 players, 107 columns
Full-Backs:            145 players, 107 columns
  - Left-Backs:        68
  - Right-Backs:       77
Defensive Midfielders: 68 players, 107 columns
Central Midfielders:   115 players, 107 columns
Attacking Midfielders: 54 players, 107 columns
Wingers:               130 players, 107 columns
  - Left Wingers:      62
  - Right Wingers:     66
Strikers:              111 players, 107 columns


In [63]:
# ============================================
# CLEAN EACH POSITION TABLE (REMOVE NULL COLUMNS)
# ============================================

In [65]:
print("\n" + "="*70)
print("CLEANING NULL COLUMNS FOR EACH POSITION")
print("="*70)

centre_backs, cb_removed = clean_position_table(centre_backs_raw, "Centre-Backs")
full_backs, fb_removed = clean_position_table(full_backs_raw, "Full-Backs")
defensive_midfielders, dm_removed = clean_position_table(defensive_midfielders_raw, "Defensive Midfielders")
central_midfielders, cm_removed = clean_position_table(central_midfielders_raw, "Central Midfielders")
attacking_midfielders, am_removed = clean_position_table(attacking_midfielders_raw, "Attacking Midfielders")
wingers, w_removed = clean_position_table(wingers_raw, "Wingers")
strikers, s_removed = clean_position_table(strikers_raw, "Strikers")


CLEANING NULL COLUMNS FOR EACH POSITION

Centre-Backs:
   Original: 149 rows, 107 columns
   Removed 100% null columns: 16
   Removed constant columns: 5
   Removed 0 empty rows
   After cleaning: 149 rows, 86 columns
   Total columns removed: 21
   Sample removed columns: ['Wing Back_Score', 'Inverted Full Back_Score', 'Defensive Full Back_Score', 'Anchor Man_Score', 'Deep Lying Playmaker_Score', 'Ball Winning Midfielder_Score', 'Mezzala_Score', 'Shadow Striker_Score', 'Playmaker_Score', 'Inside Forward_Score']...

Full-Backs:
   Original: 145 rows, 107 columns
   Removed 100% null columns: 15
   Removed constant columns: 5
   Removed 0 empty rows
   After cleaning: 145 rows, 87 columns
   Total columns removed: 20
   Removed columns: ['Aerial Stopper_Score', 'Ball Playing Defender_Score', 'Defensive Sweeper_Score', 'Anchor Man_Score', 'Deep Lying Playmaker_Score', 'Ball Winning Midfielder_Score', 'Mezzala_Score', 'Shadow Striker_Score', 'Playmaker_Score', 'Inside Forward_Score', 'Tr

In [66]:
# ============================================
# SAVE THE CLASSIFIED & CLEANED DATAFRAMES
# ============================================

In [67]:
# Save main dataframe with new Position_7 column
df.to_csv('../data/clean/classified_outfield_position_7.csv', index=False)

# Save cleaned separate position dataframes
centre_backs.to_csv('../data/clean/position_centre_backs_clean.csv', index=False)
full_backs.to_csv('../data/clean/position_full_backs_clean.csv', index=False)
defensive_midfielders.to_csv('../data/clean/position_defensive_midfielders_clean.csv', index=False)
central_midfielders.to_csv('../data/clean/position_central_midfielders_clean.csv', index=False)
attacking_midfielders.to_csv('../data/clean/position_attacking_midfielders_clean.csv', index=False)
wingers.to_csv('../data/clean/position_wingers_clean.csv', index=False)
strikers.to_csv('../data/clean/position_strikers_clean.csv', index=False)

print("\n" + "="*70)
print("✅ CLASSIFICATION & CLEANING COMPLETE! Files saved:")
print("="*70)
print("Main file:")
print("   - classified_outfield_position_7.csv (with Position_7 column)")
print("\nCleaned position files (null columns removed per position):")
print("   - position_centre_backs_clean.csv")
print("   - position_full_backs_clean.csv (includes Left-Back and Right-Back distinction)")
print("   - position_defensive_midfielders_clean.csv")
print("   - position_central_midfielders_clean.csv")
print("   - position_attacking_midfielders_clean.csv")
print("   - position_wingers_clean.csv (includes Left Winger and Right Winger distinction)")
print("   - position_strikers_clean.csv")
print("="*70)


✅ CLASSIFICATION & CLEANING COMPLETE! Files saved:
Main file:
   - classified_outfield_position_7.csv (with Position_7 column)

Cleaned position files (null columns removed per position):
   - position_centre_backs_clean.csv
   - position_full_backs_clean.csv (includes Left-Back and Right-Back distinction)
   - position_defensive_midfielders_clean.csv
   - position_central_midfielders_clean.csv
   - position_attacking_midfielders_clean.csv
   - position_wingers_clean.csv (includes Left Winger and Right Winger distinction)
   - position_strikers_clean.csv


# PLAYER RANKING SYSTEM FOR EACH POSITION

In [68]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [69]:
# ============================================
# DEFINE RANKING WEIGHTS FOR EACH POSITION
# ============================================

In [70]:
position_ranking_config = {
    'Centre-Back': {
        'primary_stats': ['Tackles (Per90)', 'Interceptions (Per90)', 'Clearances (Per90)', 'Aerial Duels Won (Per90)'],
        'secondary_stats': ['Pass Completion Rate', 'Shots Blocked (Per90)'],
        'negative_stats': ['Dribbled Past (Per90)', 'Fouls Committed (Per90)'],
        'weights': {
            'Tackles (Per90)': 0.25,
            'Interceptions (Per90)': 0.25,
            'Clearances (Per90)': 0.20,
            'Aerial Duels Won (Per90)': 0.15,
            'Pass Completion Rate': 0.10,
            'Shots Blocked (Per90)': 0.05
        }
    },
    'Full-Back': {
        'primary_stats': ['Tackles (Per90)', 'Interceptions (Per90)', 'Key Passes (Per90)', 'Successful Dribbles (Per90)'],
        'secondary_stats': ['Pass Completion Rate', 'Cross Completion Rate', 'Assists (Per90)'],
        'negative_stats': ['Dribbled Past (Per90)', 'Dispossesed (Per90)'],
        'weights': {
            'Tackles (Per90)': 0.20,
            'Interceptions (Per90)': 0.15,
            'Key Passes (Per90)': 0.20,
            'Successful Dribbles (Per90)': 0.15,
            'Pass Completion Rate': 0.10,
            'Cross Completion Rate': 0.10,
            'Assists (Per90)': 0.10
        }
    },
    'Defensive Midfield': {
        'primary_stats': ['Tackles (Per90)', 'Interceptions (Per90)', 'Pass Completion Rate', 'Key Passes (Per90)'],
        'secondary_stats': ['Aerial Duels Won (Per90)', 'Successful Dribbles (Per90)'],
        'negative_stats': ['Dispossesed (Per90)', 'Fouls Committed (Per90)'],
        'weights': {
            'Tackles (Per90)': 0.30,
            'Interceptions (Per90)': 0.20,
            'Pass Completion Rate': 0.20,
            'Key Passes (Per90)': 0.15,
            'Aerial Duels Won (Per90)': 0.10,
            'Successful Dribbles (Per90)': 0.05
        }
    },
    'Central Midfield': {
        'primary_stats': ['Pass Completion Rate', 'Key Passes (Per90)', 'Successful Dribbles (Per90)', 'Tackles (Per90)'],
        'secondary_stats': ['Assists (Per90)', 'Expected Assists (xA) (Per90)', 'Goals Scored (Per90)'],
        'negative_stats': ['Dispossesed (Per90)'],
        'weights': {
            'Pass Completion Rate': 0.25,
            'Key Passes (Per90)': 0.25,
            'Successful Dribbles (Per90)': 0.15,
            'Tackles (Per90)': 0.15,
            'Assists (Per90)': 0.10,
            'Expected Assists (xA) (Per90)': 0.05,
            'Goals Scored (Per90)': 0.05
        }
    },
    'Attacking Midfield': {
        'primary_stats': ['Key Passes (Per90)', 'Expected Assists (xA) (Per90)', 'Successful Dribbles (Per90)', 'Goals Scored (Per90)'],
        'secondary_stats': ['Pass Completion Rate', 'Assists (Per90)', 'Shot Accuracy'],
        'negative_stats': ['Dispossesed (Per90)'],
        'weights': {
            'Key Passes (Per90)': 0.30,
            'Expected Assists (xA) (Per90)': 0.20,
            'Successful Dribbles (Per90)': 0.20,
            'Goals Scored (Per90)': 0.15,
            'Pass Completion Rate': 0.10,
            'Assists (Per90)': 0.05
        }
    },
    'Winger': {
        'primary_stats': ['Successful Dribbles (Per90)', 'Key Passes (Per90)', 'Crosses (Per90)', 'Goals Scored (Per90)'],
        'secondary_stats': ['Assists (Per90)', 'Shot Accuracy', 'Cross Completion Rate'],
        'negative_stats': ['Dispossesed (Per90)'],
        'weights': {
            'Successful Dribbles (Per90)': 0.30,
            'Key Passes (Per90)': 0.20,
            'Crosses (Per90)': 0.15,
            'Goals Scored (Per90)': 0.15,
            'Assists (Per90)': 0.10,
            'Shot Accuracy': 0.05,
            'Cross Completion Rate': 0.05
        }
    },
    'Striker': {
        'primary_stats': ['Goals Scored (Per90)', 'Shots On Target (Per90)', 'Shot Accuracy', 'Expected Goals (xG) (Per90)'],
        'secondary_stats': ['Assists (Per90)', 'Key Passes (Per90)', 'Successful Dribbles (Per90)'],
        'negative_stats': ['Offsides (Per90)', 'Dispossesed (Per90)'],
        'weights': {
            'Goals Scored (Per90)': 0.40,
            'Shots On Target (Per90)': 0.20,
            'Shot Accuracy': 0.15,
            'Expected Goals (xG) (Per90)': 0.10,
            'Assists (Per90)': 0.10,
            'Key Passes (Per90)': 0.03,
            'Successful Dribbles (Per90)': 0.02
        }
    }
}

In [71]:
def rank_players(df, position_name, config):
    """
    Rank players based on position-specific stats
    """
    if df.empty:
        print(f"No players found for {position_name}")
        return df
    
    df_ranked = df.copy()
    
    # Initialize score column
    df_ranked['Performance_Score'] = 0.0
    
    # Track which stats were used
    stats_used = []
    weights_used = []
    
    # Calculate weighted score for each stat
    for stat, weight in config['weights'].items():
        if stat in df_ranked.columns:
            # Convert to numeric if needed
            df_ranked[stat] = pd.to_numeric(df_ranked[stat], errors='coerce')
            
            # Normalize the stat (0 to 100 scale)
            valid_mask = df_ranked[stat].notna()
            if valid_mask.any():
                min_val = df_ranked.loc[valid_mask, stat].min()
                max_val = df_ranked.loc[valid_mask, stat].max()
                
                if max_val > min_val:
                    # Normalize to 0-100 scale
                    normalized = (df_ranked[stat] - min_val) / (max_val - min_val) * 100
                else:
                    normalized = 50  # Default if all values are the same
                
                # Apply weight
                df_ranked['Performance_Score'] += normalized.fillna(0) * weight
                stats_used.append(stat)
                weights_used.append(weight)
    
    # Apply negative stat penalties (if any)
    for neg_stat in config.get('negative_stats', []):
        if neg_stat in df_ranked.columns:
            df_ranked[neg_stat] = pd.to_numeric(df_ranked[neg_stat], errors='coerce')
            valid_mask = df_ranked[neg_stat].notna()
            if valid_mask.any():
                # Higher negative stat = lower score (inverse normalization)
                min_val = df_ranked.loc[valid_mask, neg_stat].min()
                max_val = df_ranked.loc[valid_mask, neg_stat].max()
                
                if max_val > min_val:
                    # Inverse normalization (higher value = lower score)
                    normalized = (1 - (df_ranked[neg_stat] - min_val) / (max_val - min_val)) * 100
                else:
                    normalized = 50
                
                # Apply penalty (10% reduction maximum)
                df_ranked['Performance_Score'] -= normalized.fillna(0) * 0.05
    
    # Ensure score is between 0 and 100
    df_ranked['Performance_Score'] = df_ranked['Performance_Score'].clip(0, 100)
    
    # Add rank
    df_ranked['Rank'] = df_ranked['Performance_Score'].rank(method='min', ascending=False).astype(int)
    
    # Sort by rank
    df_ranked = df_ranked.sort_values('Rank')
    
    # Print ranking summary
    print(f"\n{position_name} Ranking Summary:")
    print(f"   Players ranked: {len(df_ranked)}")
    print(f"   Stats used: {stats_used}")
    print(f"   Avg Score: {df_ranked['Performance_Score'].mean():.2f}")
    print(f"   Top Score: {df_ranked['Performance_Score'].max():.2f}")
    
    return df_ranked

In [72]:
# ============================================
# LOAD ALL CLEANED POSITION FILES
# ============================================

In [73]:
print("="*70)
print("LOADING CLEANED POSITION FILES")
print("="*70)

# Load the cleaned position files
position_files = {
    'Centre-Back': '../data/clean/position_centre_backs_clean.csv',
    'Full-Back': '../data/clean/position_full_backs_clean.csv',
    'Defensive Midfield': '../data/clean/position_defensive_midfielders_clean.csv',
    'Central Midfield': '../data/clean/position_central_midfielders_clean.csv',
    'Attacking Midfield': '../data/clean/position_attacking_midfielders_clean.csv',
    'Winger': '../data/clean/position_wingers_clean.csv',
    'Striker': '../data/clean/position_strikers_clean.csv'
}

position_dfs = {}
for pos_name, file_path in position_files.items():
    try:
        df = pd.read_csv(file_path)
        position_dfs[pos_name] = df
        print(f"✓ Loaded {pos_name}: {len(df)} players, {len(df.columns)} columns")
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        position_dfs[pos_name] = pd.DataFrame()  # Empty dataframe

LOADING CLEANED POSITION FILES
✓ Loaded Centre-Back: 149 players, 86 columns
✓ Loaded Full-Back: 145 players, 87 columns
✓ Loaded Defensive Midfield: 68 players, 86 columns
✓ Loaded Central Midfield: 115 players, 83 columns
✓ Loaded Attacking Midfield: 54 players, 83 columns
✓ Loaded Winger: 130 players, 86 columns
✓ Loaded Striker: 111 players, 86 columns


In [74]:
# ============================================
# RANK PLAYERS FOR EACH POSITION
# ============================================

In [75]:
print("\n" + "="*70)
print("RANKING PLAYERS BY POSITION")
print("="*70)

ranked_positions = {}
for pos_name, df in position_dfs.items():
    if not df.empty and pos_name in position_ranking_config:
        ranked_df = rank_players(df, pos_name, position_ranking_config[pos_name])
        ranked_positions[pos_name] = ranked_df
    elif df.empty:
        print(f"\n{pos_name}: No data to rank")
    else:
        print(f"\n{pos_name}: No ranking configuration found")


RANKING PLAYERS BY POSITION

Centre-Back Ranking Summary:
   Players ranked: 149
   Stats used: ['Tackles (Per90)', 'Interceptions (Per90)', 'Clearances (Per90)', 'Aerial Duels Won (Per90)', 'Pass Completion Rate', 'Shots Blocked (Per90)']
   Avg Score: 16.39
   Top Score: 73.63

Full-Back Ranking Summary:
   Players ranked: 145
   Stats used: ['Tackles (Per90)', 'Interceptions (Per90)', 'Key Passes (Per90)', 'Successful Dribbles (Per90)', 'Pass Completion Rate', 'Cross Completion Rate', 'Assists (Per90)']
   Avg Score: 23.69
   Top Score: 52.59

Defensive Midfield Ranking Summary:
   Players ranked: 68
   Stats used: ['Tackles (Per90)', 'Interceptions (Per90)', 'Pass Completion Rate', 'Key Passes (Per90)', 'Aerial Duels Won (Per90)', 'Successful Dribbles (Per90)']
   Avg Score: 26.61
   Top Score: 50.00

Central Midfield Ranking Summary:
   Players ranked: 115
   Stats used: ['Pass Completion Rate', 'Key Passes (Per90)', 'Successful Dribbles (Per90)', 'Tackles (Per90)', 'Assists (Per

In [76]:
# ============================================
# DISPLAY TOP 5 PLAYERS PER POSITION
# ============================================

In [77]:
print("\n" + "="*70)
print("TOP 5 PLAYERS PER POSITION")
print("="*70)

for pos_name, ranked_df in ranked_positions.items():
    if not ranked_df.empty:
        print(f"\n{'='*70}")
        print(f"🏆 TOP 5 {pos_name.upper()}S")
        print(f"{'='*70}")
        
        # Select columns to display
        display_cols = ['Name', 'Team', 'Age', 'Market Value', 'Performance_Score', 'Rank']
        
        # Add position-specific top stats
        if pos_name == 'Striker':
            display_cols.extend(['Goals Scored (Per90)', 'Shot Accuracy'])
        elif pos_name == 'Winger':
            display_cols.extend(['Successful Dribbles (Per90)', 'Key Passes (Per90)'])
        elif pos_name == 'Centre-Back':
            display_cols.extend(['Tackles (Per90)', 'Interceptions (Per90)'])
        elif pos_name == 'Full-Back':
            display_cols.extend(['Tackles (Per90)', 'Key Passes (Per90)'])
        elif 'Midfield' in pos_name:
            display_cols.extend(['Key Passes (Per90)', 'Pass Completion Rate'])
        
        # Filter to columns that exist
        display_cols = [col for col in display_cols if col in ranked_df.columns]
        
        # Display top 5
        print(ranked_df[display_cols].head(5).to_string(index=False))
        
        # Also show bottom 5 (optional)
        print(f"\n📉 BOTTOM 5 {pos_name.upper()}S:")
        print(ranked_df[display_cols].tail(5).to_string(index=False))


TOP 5 PLAYERS PER POSITION

🏆 TOP 5 CENTRE-BACKS
            Name                   Team  Age  Market Value  Performance_Score  Rank  Tackles (Per90)  Interceptions (Per90)
     Ville Koski       Deportivo Alavés   24     1500000.0          73.630000     1            16.36                   8.18
Federico Gattoni             Sevilla FC   27     1000000.0          35.000000     2             0.00                   0.00
    Miguel Rubio RCD Espanyol Barcelona   28     2000000.0          31.934992     3             1.83                   0.91
      Kike Salas             Sevilla FC   23     8000000.0          26.862937     4             2.29                   1.95
     Kevin Danso      Tottenham Hotspur   27    20000000.0          25.968652     5             1.97                   1.22

📉 BOTTOM 5 CENTRE-BACKS:
              Name                    Team  Age  Market Value  Performance_Score  Rank  Tackles (Per90)  Interceptions (Per90)
       David López               Girona FC   36      

In [78]:
# ============================================
# ADDITIONAL: RANK WITH VALUE FOR MONEY (Age + Market Value)
# ============================================

In [79]:
def rank_value_for_money(df, position_name):
    """
    Rank players by value for money (performance relative to cost)
    """
    if df.empty:
        return df
    
    df_value = df.copy()
    
    # Normalize Market Value (lower = better for value)
    if 'Market Value' in df_value.columns:
        df_value['Market Value'] = pd.to_numeric(df_value['Market Value'], errors='coerce')
        valid_mask = df_value['Market Value'].notna()
        if valid_mask.any():
            min_val = df_value.loc[valid_mask, 'Market Value'].min()
            max_val = df_value.loc[valid_mask, 'Market Value'].max()
            if max_val > min_val:
                df_value['Value_Normalized'] = (df_value['Market Value'] - min_val) / (max_val - min_val)
            else:
                df_value['Value_Normalized'] = 0.5
    
    # Normalize Age (younger = better for value)
    if 'Age' in df_value.columns:
        df_value['Age'] = pd.to_numeric(df_value['Age'], errors='coerce')
        valid_mask = df_value['Age'].notna()
        if valid_mask.any():
            min_val = df_value.loc[valid_mask, 'Age'].min()
            max_val = df_value.loc[valid_mask, 'Age'].max()
            if max_val > min_val:
                df_value['Age_Normalized'] = (df_value['Age'] - min_val) / (max_val - min_val)
            else:
                df_value['Age_Normalized'] = 0.5
    
    # Value for Money Score = Performance_Score - (Value_Normalized * 30) - (Age_Normalized * 20)
    if 'Performance_Score' in df_value.columns:
        df_value['Value_Score'] = df_value['Performance_Score']
        if 'Value_Normalized' in df_value.columns:
            df_value['Value_Score'] -= df_value['Value_Normalized'] * 30
        if 'Age_Normalized' in df_value.columns:
            df_value['Value_Score'] -= df_value['Age_Normalized'] * 20
        
        df_value['Value_Score'] = df_value['Value_Score'].clip(0, 100)
        df_value['Value_Rank'] = df_value['Value_Score'].rank(method='min', ascending=False).astype(int)
        df_value = df_value.sort_values('Value_Rank')
    
    print(f"\n{position_name} - Best Value for Money:")
    display_cols = ['Name', 'Team', 'Age', 'Market Value', 'Performance_Score', 'Value_Score', 'Value_Rank']
    display_cols = [col for col in display_cols if col in df_value.columns]
    print(df_value[display_cols].head(5).to_string(index=False))
    
    return df_value

In [80]:
# ============================================
# CALCULATE VALUE FOR MONEY RANKINGS
# ============================================

In [81]:
print("\n" + "="*70)
print("BEST VALUE FOR MONEY (Performance vs Cost)")
print("="*70)

value_rankings = {}
for pos_name, ranked_df in ranked_positions.items():
    if not ranked_df.empty:
        value_df = rank_value_for_money(ranked_df, pos_name)
        value_rankings[pos_name] = value_df


BEST VALUE FOR MONEY (Performance vs Cost)

Centre-Back - Best Value for Money:
            Name                   Team  Age  Market Value  Performance_Score  Value_Score  Value_Rank
     Ville Koski       Deportivo Alavés   24     1500000.0          73.630000    67.379343           1
Federico Gattoni             Sevilla FC   27     1000000.0          35.000000    25.387342           2
    Miguel Rubio RCD Espanyol Barcelona   28     2000000.0          31.934992    20.811042           3
      Kike Salas             Sevilla FC   23     8000000.0          26.862937    19.612412           4
  Tanguy Nianzou             Sevilla FC   23     1400000.0          20.399213    15.358509           5

Full-Back - Best Value for Money:
         Name                    Team  Age  Market Value  Performance_Score  Value_Score  Value_Rank
  Héctor Fort                Elche CF   19    12000000.0          45.977016    40.941148           1
 João Cancelo            FC Barcelona   31     9000000.0        

In [82]:
# ============================================
# SAVE RANKED DATAFRAMES
# ============================================

In [84]:
print("\n" + "="*70)
print("SAVING RANKED POSITION FILES")
print("="*70)

for pos_name, ranked_df in ranked_positions.items():
    if not ranked_df.empty:
        filename = f"../data/clean/ranked_{pos_name.lower().replace(' ', '_')}s.csv"
        ranked_df.to_csv(filename, index=False)
        print(f"✓ Saved: {filename}")

# Save value rankings
for pos_name, value_df in value_rankings.items():
    if not value_df.empty:
        filename = f"../data/clean/value_ranked_{pos_name.lower().replace(' ', '_')}s.csv"
        value_df.to_csv(filename, index=False)
        print(f"✓ Saved: {filename}")


SAVING RANKED POSITION FILES
✓ Saved: ../data/clean/ranked_centre-backs.csv
✓ Saved: ../data/clean/ranked_full-backs.csv
✓ Saved: ../data/clean/ranked_defensive_midfields.csv
✓ Saved: ../data/clean/ranked_central_midfields.csv
✓ Saved: ../data/clean/ranked_attacking_midfields.csv
✓ Saved: ../data/clean/ranked_wingers.csv
✓ Saved: ../data/clean/ranked_strikers.csv
✓ Saved: ../data/clean/value_ranked_centre-backs.csv
✓ Saved: ../data/clean/value_ranked_full-backs.csv
✓ Saved: ../data/clean/value_ranked_defensive_midfields.csv
✓ Saved: ../data/clean/value_ranked_central_midfields.csv
✓ Saved: ../data/clean/value_ranked_attacking_midfields.csv
✓ Saved: ../data/clean/value_ranked_wingers.csv
✓ Saved: ../data/clean/value_ranked_strikers.csv


In [85]:
print("\n" + "="*70)
print("MASTER RANKING - TOP 50 PLAYERS OVERALL")
print("="*70)

# Combine all ranked players
all_ranked = []
for pos_name, ranked_df in ranked_positions.items():
    if not ranked_df.empty:
        temp_df = ranked_df.copy()
        temp_df['Position'] = pos_name
        all_ranked.append(temp_df)

if all_ranked:
    master_ranking = pd.concat(all_ranked, ignore_index=True)
    master_ranking = master_ranking.sort_values('Performance_Score', ascending=False)
    master_ranking['Overall_Rank'] = range(1, len(master_ranking) + 1)
    
    # Display top 50
    display_cols = ['Overall_Rank', 'Name', 'Team', 'Position', 'Age', 'Market Value', 'Performance_Score']
    display_cols = [col for col in display_cols if col in master_ranking.columns]
    print(master_ranking[display_cols].head(50).to_string(index=False))
    
    # Save master ranking
    master_ranking.to_csv('../data/clean/master_player_ranking.csv', index=False)
    print(f"\n✓ Saved master ranking: master_player_ranking.csv ({len(master_ranking)} players)")

print("\n" + "="*70)
print("RANKING COMPLETE!")
print("="*70)


MASTER RANKING - TOP 50 PLAYERS OVERALL
 Overall_Rank                   Name                    Team           Position  Age  Market Value  Performance_Score
            1            Ville Koski        Deportivo Alavés        Centre-Back   24     1500000.0          73.630000
            2          Jamie Gittens              Chelsea FC             Winger   21    35000000.0          62.449576
            3        Bruno Guimarães        Newcastle United   Central Midfield   28    75000000.0          62.164762
            4           Lamine Yamal            FC Barcelona             Winger   18   200000000.0          61.559470
            5              Marc Guiu              Chelsea FC            Striker   20    12000000.0          60.927493
            6                  Pedri            FC Barcelona   Central Midfield   23   150000000.0          58.289197
            7                   Isco     Real Betis Balompié Attacking Midfield   33     4000000.0          57.986992
            8  